<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_WEEK03_2_Code_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes Prediction Challenge

[link text](https://www.kaggle.com/code/jagrutiyuvrajdhangar/diabetes-prediction-4)

## 코드 흐름 요약

이 대회 코드는 당뇨병 여부를 예측하는 이진 분류 모델을 만들고, 여러 부스팅 계열 모델의 예측을 앙상블하고 있음

1. 라이브러리 불러오기
- numpy, pandas 같은 기본 라이브러리
- 교차검증용 StratifiedKFold
- 평가 지표 roc_auc_score
- 결측치 처리 SimpleImputer
- 스케일링 StandardScaler
- 부스팅 모델 XGBoost, LightGBM, CatBoost
- 불균형 데이터 보정용 SMOTE

2. 데이터 불러오기 및 train+test 합친 전처리
- full = pd.concat([X, test], axis=0)로 합침
  - 학습 데이터와 테스트 데이터에 같은 방식으로 피처 엔지니어링과 범주형 처리를 적용하기 위해서 합침

3. 파생 변수 생성
- BMI_Age = BMI × Age
- Glucose_Insulin = Glucose × Insulin

> 즉, 기존 변수끼리 곱해서 새로운 정보를 만들어냈는데, 이러한 방식은 단순히 원래 변수만 쓰는 것보다 모델이 관계를 더 잘 잡게 도와줄 수 있음

4. 범주형 변수 숫자화
- object 타입 컬럼이 있으면 category code로 변환
- 머신러닝 모델은 문자열을 직접 처리하지 못하므로 숫자로 바꾸는 과정이 필요함

5. 다시 train, test 분리 후 결측 처리
- SimpleImputer(strategy="median")
- 결측값을 중앙값으로 채움
- 중앙값은 이상치에 덜 민감해서 자주 사용됨
- 스케일링
  - StandardScaler()로 평균 0, 표준편차 1 형태로 변환

6. 교차검증 준비
- StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 타깃의 클래스 비율을 유지하면서 5개 폴드로 나눔
- 당뇨병 예측처럼 클래스 불균형이 있을 수 있는 문제에서 중요함

7. 모델 정의
- XGBClassifier, LGBMClassifier, CatBoostClassifier
- 즉, 서로 다른 세 개의 부스팅 모델을 사용해서 성능을 높임

8. SMOTE 적용 후 각 폴드별 학습
- **각 fold에서 train 부분에만 SMOTE를 적용**
- 소수 클래스 데이터를 인공적으로 늘려 클래스 불균형 문제를 완화함
- 이후, 각 모델을 학습시킨 뒤 validation 예측값과 test 예측값을 저장함

9. AUC 계산
- fold마다 AUC를 출력
- 전체 OOF(out-of-fold) 예측으로 다시 AUC 계산
- 최종적으로 세 모델의 평균 예측값으로 FINAL CV AUC를 계산

## 새롭게 알게 된 내용 및 배울 점

**- SMOTE?**
  - Synthetic Minority Over-Sampling
  > 개수가 적은 클래스의 데이터를 인위적으로 생성하여 클래스 불균형을 완화하는 방법
  - 예를 들어, 0이 900개, 1이 100개 있다고 했을 때, 이 데이터를 학습한 모델을 0이라고 예측하는 것이 훨씬 안전하다고 학습함, 따라서 적은 쪽 클래스 데이터를 늘려주는 것임
  - 단순 복사가 아니라 기존 소수 클래스 샘플들 사이를 보간(interpolation)해서 새로운 점을 만들어 냄
  - 반드시 학습용 데이터에만 적용 !!
  - 연속형 수치 데이터에 더 자연스럽기에 범주형이 많으면 부적절할 수도 있음
  > 트리 모델에서의 불균형 처리 방법
    - class_weight='balanced'
    - XGBoost의 scale_pos_weight
    - threshold tuning 등

- smote는 전체 데이터에 적용하면 안되고, fold 안에서만 적용해야 함
  - 해당 코드에서는 교차검증 시, 먼저 train/valid를 나누고 그 다음에 train fold에만 smote를 적용함
  - 이렇게 해야 validation 데이터 정보가 train에 새어나가지 않음
  - 만약 전체 데이터에 smote를 먼저 적용한 뒤 cv를 하면 data leakage 문제가 생길 수 있음


**- OOF?**
  - Out-Of-Fold
  > 자신이 속하지 않은 fold에서 학습된 모델로 예측된 값을 모아둔 것
  - 각 샘플을 학습에 사용하지 않도록 검증용으로 빼두고 예측을 생성하면 각 샘플은 한 번씩 validation이 됨
  - 자기 자신을 본 적 없는 모델이 예측하고 그래서 train score보다 덜 낙관적이며 실제 test 성능과 더 비슷할 가능성이 높음
  > 모델의 일반화 성능을 잘 보여주는 지표

- OOF 예측을 이용하면 모델 성능을 더 공정하게 볼 수 있음
  - ```
    fold_oof[val_idx] = model.predict_proba(X_valid)[:,1]
    ```
  - 이렇게 하면 각 샘플은 자기 자신을 학습에 사용하지 않은 모델로 예측된 값을 가지게 됨
  - 이걸 모아서 계산한 AUC는 단순 train score보다 신뢰도가 더 높음




< 파생 변수 생성 >
```
full["BMI_Age"] = full["BMI"] * full["Age"]
full["Glucose_Insulin"] = full["Glucose"] * full["Insulin"]
```
- BMI와 Age가 함께 당뇨병 위험에 영향을 줄 수 있다는 가정을 데이터에 반영한 것

< 결측 처리 >
```
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns)
```
- fit_transform은 train에 적용해서 기준을 학습

< smote >
```
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
```
- train fold 안에서 소수 클래스를 증강
- 데이터 불균형 문제를 완화
- validation에는 적용하지 않는다는 점이 핵심


< oof 성능 평가 >
```
fold_oof[val_idx] = model.predict_proba(X_valid)[:,1]
fold_pred += model.predict_proba(X_test)[:,1] / folds.n_splits
```
- validation에서는 확률 예측값을 저장해서 OOF 성능 평가에 사용
- test는 각 fold의 예측을 평균내어 더 안정적인 결과를 만듦
